# 参数与扫描

**从这里开始。** 需要会逐格运行 Notebook、保存 Python 文件。完成后，你能预览并运行一次扫描，在 Notebook 和应用 Runs 中查看同一结果，再把 shots 从 64 改成 32，比较新旧运行。做到这里即可开始改写自己的实验；后面的新增参数表是选学。想继续写源码可选 refresh；计算和分组分析按需选择，校准不是入门必修。

这是纯软件合成响应，不连接设备，也不是物理校准。Help 已把普通作者目录和 Python 环境登记到当前应用。
在外部编辑器打开本文件，选择此目录的 `.venv` 内核。应用保存运行和历史，编辑器保存源码和 Notebook。

In [ ]:
import scopecat as sc

session = sc.notebook()
session

In [ ]:
import numpy as np
from my_experiment.setup import open_parameters
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)

这份练习有独立参数分支。打开已有分支保留你保存的修改，不会覆盖其他练习或真实参数。
下格只保存参数、构造扫描并预览，不采集。可自行修改 `params[Drive]["q0"].frequency`（GHz）后再保存。

In [ ]:
from my_experiment.parameters import Drive

print("频率（GHz）:", params[Drive]["q0"].frequency)
saved = params.save(note="参数沙盒")
request = teaching_rabi().sweep(amplitude=np.linspace(0, 0.8, 7))
prepared = session.prepare(request, parameters=params)
print("预览点数:", prepared.preview.point_count)
assert prepared.preview.point_count == 7

**新采集：只有执行下一格才开始一次新运行。** 每点默认保存 64 个 shot。再次执行会新建运行；只想查看已有结果时，跳过此格，使用下面的结果格或应用 Runs。

In [ ]:
run = prepared.run().wait(timeout=120).result()

**查看结果：下格只读取已有运行，可反复执行，不会采集。**
重启内核后，先执行连接应用和打开参数的两格。可直接从应用 **Runs** 打开旧运行；若要在 Notebook 继续分析，从 Runs 详情复制 run ID，先执行 `run = session.run("你的 run ID")`，再执行下格。不要为了恢复结果选择 Run All。

In [ ]:
shots = np.asarray(run.measurements()["iq"].require_values())
print("扫描点 × shots:", shots.shape)
print("run:", run.id)
print("每个扫描点的平均 IQ:", shots.mean(axis=1))

需要在桌面查看时，运行下一格。它打开同一应用的既有对象，不采集；请先保持应用桌面运行。若未确认打开，先检查已有窗口，再用异常的 `request.wait()` 重试同一请求，避免重新打开。

In [ ]:
run.open()

设置 `request.values["shots"] = 32`，再执行 `prepared = session.prepare(request, parameters=params)`，然后明确执行采集格和结果格。无需改源码或刷新；新结果应为 7 × 32，旧运行仍保留 7 × 64。不要先重跑构造请求的格子，否则会重新使用默认值。
再设置 `request.values["amplitude"] = sc.Scan(np.linspace(0, 0.8, 5))`，重新准备并确认预览为 5 点，明确采集后得到 5 × 32。若要练习源码编辑，可另行修改 `src/my_experiment/teaching.py` 的默认值并保存，然后构造新请求。已有请求保留原来源，修改请求值也不会改变已经准备的实验。
关闭编辑器不停止应用。下次从 Help 继续，重新连接并打开参数即可；打开 Notebook 和 Runs 不会自动重新采集。

参数声明在 `src/my_experiment/parameters.py`；实验在 `teaching.py`，初始化在 `setup.py`。它们都是本项目代码，可以查看和修改。

下面声明一张独立表，与 Drive 共存。同名表不能被当作另一张表；改变已有列的单位或主键需要显式结构操作。先保存已有值修改，再声明新表。

In [ ]:
class Readout(sc.ParameterModel, table="my_readout"):
    id: sc.Param[str] = sc.param(key=True)
    frequency: sc.Magnitude[float] = sc.quantity(unit="GHz")


params.declare_table(Readout)
if "q0" not in params[Readout]:
    params[Readout].add(Readout(id="q0", frequency=5.146))
params.save(note="自己的参数表")
params

小扩展：把 Readout 声明移到本地 `parameters.py`，将实验中的 `sc.parameter_ref(Drive.frequency, "q0")` 替换为 `sc.parameter_ref(Readout.frequency, "q0")`，并在实验模块中 import Readout。新增表不会自动接入实验，只有显式使用的参数才影响结果。

Notebook 默认工作区只为新请求选择已保存代码；旧请求和预览保留原版本。初始化只设置新表的起点，不会覆盖你已保存的参数。